# SENTRY + PSID-8 — Full Reproduction Notebook (Kaggle, 2×T4)

Reproduces every number reported in the paper *"Towards Physical Security Incident
Detection in Video Surveillance: The SENTRY Architecture and the Data Gap"*.

**Pipeline**
| Cell | Produces |
|---|---|
| [0]–[1] | Environment pinning, module import, configuration |
| [2] | Le2i → clips, camera-disjoint splits, **Phase-1 freeze** (hashes) |
| [3] | **Stage A**: frame-level baseline, 3 pre-registered seeds → mAP50 |
| [4] | SENTRY assembly + parameter overhead |
| [5] | **Stage B**: supervised temporal fine-tuning of the TFM |
| [6] | **Level S**: SENTRY vs frame-level, event F1 (epoch ablation) |
| [7] | **Level E**: UCF-Crime zero-shot, 3 prompt conditions |
| [8] | Latency: frame-level vs SENTRY |
| [9] | Artifact export |

**Required Kaggle datasets**: Le2i fall, UCF-Crime (full, with videos), and this
repository uploaded as a Dataset named `sentry-psid8`. Settings: GPU T4×2,
Internet ON.

**Integrity protocol** (`protocol/PREREGISTRATION.md`): splits are frozen by hash
before any training; the test split is evaluated once per finalized model.

**Runtime budget**: Stage A ≈ 3h30 per seed; Level E ≈ 35 min per prompt
condition. Run long cells via *Save & Run All (Commit)*, not interactively.

In [ ]:
# %% [0] Environment — pin versions and remove the Ray Tune conflict
# ultralytics auto-registers a Ray Tune callback; the `ray` build shipped on
# Kaggle removed the internal symbol it calls, crashing training at
# on_fit_epoch_end. We do not use Ray, so removing it is the robust fix.
!pip uninstall -y ray 2>/dev/null | tail -1
!pip -q install "ultralytics==8.3.49"

import os, sys, json, glob, time, random, math
import numpy as np
import torch, ultralytics

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "n_gpus": torch.cuda.device_count(),
    "seed": SEED,
}
json.dump(VERSIONS, open("versions.json", "w"), indent=2)
print(json.dumps(VERSIONS, indent=2))
print("\nIMPORTANT: restart the kernel once after the ray uninstall, then re-run.")

## [1] Import the repository modules and set configuration

Upload the repo zip as a Kaggle Dataset named `sentry-psid8`. Confirm the mount
path with `!ls /kaggle/input` — Kaggle mounts either as
`/kaggle/input/<slug>/` or `/kaggle/input/datasets/<user>/<slug>/` depending on
the account. Set `REPO` to whatever that listing shows.

In [ ]:
# %% [1] Repo import + configuration
!ls /kaggle/input

REPO = "/kaggle/input/datasets/salomopena/sentry-psid8/sentry-psid8"   # ADJUST
assert os.path.isdir(REPO), f"Repo not found at {REPO}; check `ls /kaggle/input`"
sys.path.insert(0, REPO)

import sentry
from sentry.modules import TemporalFeatureMemory
from sentry.tubes import TubeLinker, confirm_events
from sentry.metrics import event_auc, event_prf, bootstrap_ci, streams_per_gpu
from sentry.seeds import run_over_seeds, dataloader_seeding, set_all_seeds, DECLARED_SEEDS
from sentry.data import VideoClipDataset, collate_clips, collate_batched
from sentry.ultralytics_adapter import SentryYOLO
from sentry.stageb_train import build_loss_batch, temporal_consistency_feats
from sentry.train import build_criterion, run_stage_b
print(f"repo modules OK — sentry-psid8 v{sentry.__version__}")   # expect >= 0.3.0

CFG = dict(
    # --- dataset roots: verify each against `ls /kaggle/input` ---
    le2i_root    = "/kaggle/input/datasets/tuyenldvn/falldataset-imvia",
    ucf_root     = "/kaggle/input/datasets/minmints/ufc-crime-full-dataset",
    ucf_temporal = "/kaggle/input/datasets/minmints/ufc-crime-full-dataset/"
                   "Temporal_Anomaly_Annotation_for_Testing_Videos.txt",
    # --- model / training ---
    imgsz        = 640,
    model_base   = "yolov8m.pt",
    epochs_A     = 100,       # early-stopped by patience
    epochs_B     = 10,        # Stage B ablation used {5, 10, 30}
    stageB_batch = 2,          # clips batched together per timestep (see ARCHITECTURE.md 7.1)
    window       = 8,         # W
    hidden_ch    = 128,       # C_h of the TFM
    lambda_tc    = 1.0,       # lambda_4 of Eq. (3)
    # --- Level E ---
    ucf_per_class = 30,       # 30/class -> 110 videos (the reported run)
    fps_stride    = 5,        # evaluate 1 of every 5 frames
)
ORDER = ["accident", "suspicious_behavior", "crime", "fire",
         "intrusion", "suspicious_object", "fall", "vandalism"]   # class ids 0..7
FALL = 6
print(json.dumps({k: v for k, v in CFG.items()}, indent=1))

## [2] Level S data — Le2i → clips, splits, Phase-1 freeze

Run the converter with `--preview` FIRST and inspect the images: Le2i mirrors
differ in the per-frame bbox column order. If the drawn box does not cover the
falling person, switch to `--bbox-format center_hw`.

The converter handles three quirks of the public mirrors: doubly nested folders
(`Coffee_room_02/Coffee_room_02/`), the misspelled `Annotations_files`, and AVI
files whose corrupt MP3 track crashes OpenCV (decoding goes through the `ffmpeg`
CLI with `-an`). It also applies the **dummy-interval rule**: a clip whose fall
interval contains no annotated box (e.g. `[1,2]`) is an ADL negative, not a fall.

In [ ]:
# %% [2] Le2i conversion → splits → freeze
BBOX_FMT = "corners"     # switch to "center_hw" if the preview boxes look wrong

# 2.1 preview (inspect, then comment out)
!python {REPO}/psid8/scripts/le2i_to_clips.py --root {CFG['le2i_root']} \
    --out /kaggle/working/le2i --preview 6 --bbox-format {BBOX_FMT}

from IPython.display import Image, display
for f in sorted(glob.glob("/kaggle/working/preview_*.jpg")) + sorted(glob.glob("preview_*.jpg")):
    display(Image(f))

In [ ]:
# %% [2.2] Full conversion + splits + integrity freeze (run after the preview)
!rm -rf /kaggle/working/le2i
!python {REPO}/psid8/scripts/le2i_to_clips.py --root {CFG['le2i_root']} \
    --out /kaggle/working/le2i --bbox-format {BBOX_FMT}
!python {REPO}/psid8/scripts/build_splits.py /kaggle/working/le2i/manifest.json \
    --out /kaggle/working/le2i/splits.json --seed 0
!cd /kaggle/working/le2i && python {REPO}/psid8/scripts/integrity_check.py \
    manifest.json splits.json

# Reported run: 130 clips (96 falls, 34 negatives); 4 cameras; mode=global_fallback
# (scenario stratification is infeasible with 2 cameras/scenario — declared in the
# paper); splits train:78 / val:30 / test:22 with test = Coffee_room_02 (unseen).
print(open("/kaggle/working/le2i/integrity_report.json").read())

In [ ]:
# %% [2.3] Ultralytics layout for Level S
# Train keeps ALL labeled frames and 1 of every `bg_stride_train` background
# frames; val/test keep everything — metrics are never subsampled.
import yaml, shutil

def make_yolo_layout(clips_root, splits_json, out_root, bg_stride_train=5):
    sp = json.load(open(splits_json))["splits"]
    shutil.rmtree(out_root, ignore_errors=True)
    for split, ids in sp.items():
        for sub in ("images", "labels"):
            os.makedirs(f"{out_root}/{sub}/{split}", exist_ok=True)
        bg_seen = 0
        for cid in ids:
            for fp in sorted(glob.glob(f"{clips_root}/clips/{cid}/frames/*.jpg")):
                lp = fp.replace("/frames/", "/labels/").replace(".jpg", ".txt")
                labeled = os.path.exists(lp) and os.path.getsize(lp) > 0
                if split == "train" and not labeled:
                    bg_seen += 1
                    if (bg_seen - 1) % bg_stride_train:
                        continue
                stem = f"{cid}__{os.path.basename(fp)}"
                os.symlink(fp, f"{out_root}/images/{split}/{stem}")
                dst = f"{out_root}/labels/{split}/{stem.replace('.jpg', '.txt')}"
                if labeled: os.symlink(lp, dst)
                else: open(dst, "w").close()
    # nc=8: Le2i labels carry the canonical class id 6 (fall); ids must be < nc
    names = {i: n for i, n in enumerate(ORDER)}
    yaml.safe_dump({"path": out_root, "train": "images/train", "val": "images/val",
                    "test": "images/test", "names": names},
                   open(f"{out_root}/data.yaml", "w"))
    for split in sp:
        n = len(glob.glob(f"{out_root}/images/{split}/*.jpg"))
        pos = sum(1 for l in glob.glob(f"{out_root}/labels/{split}/*.txt")
                  if os.path.getsize(l) > 0)
        print(f"{split}: {n} frames | {pos} with a fall label")
    return f"{out_root}/data.yaml"

DATA_FALL = make_yolo_layout("/kaggle/working/le2i",
                             "/kaggle/working/le2i/splits.json",
                             "/kaggle/working/fall_yolo")
# Reported run: train 5778 / val 7322 / test 13141 frames.
# NOTE: symlinks do not survive Kaggle version archiving — re-run this cell
# (seconds) whenever you resume in a fresh session.

## [3] Stage A — frame-level baseline, 3 pre-registered seeds

Seeds `[0, 1, 2]` are declared **before** any result is seen (anti seed-hacking).
Splits stay fixed across seeds, so the reported variance isolates training
stochasticity. `run_over_seeds` is resumable at the seed level and `resume=True`
at the epoch level, so an expired Kaggle session costs nothing.

`device=0` is deliberate: multi-GPU DDP spawns a subprocess that hangs inside
notebooks. **Reported result: mAP50 = 0.675 ± 0.043, CI95 [0.627, 0.711].**

In [ ]:
# %% [3] Stage A — multi-seed, resumable
from ultralytics import YOLO

def stage_a_pipeline(seed):
    run_dir = f"runs/stageA_fall_s{seed}"
    last = f"{run_dir}/weights/last.pt"
    if os.path.exists(last):
        print(f"[seed {seed}] resuming from {last}")
        YOLO(last).train(resume=True)
    else:
        YOLO(CFG["model_base"]).train(
            data=DATA_FALL, imgsz=CFG["imgsz"], epochs=CFG["epochs_A"],
            seed=seed, device=0, batch=16, workers=2, cache=False,
            patience=20, project="runs", name=f"stageA_fall_s{seed}",
            deterministic=True, exist_ok=True)
    m = YOLO(f"{run_dir}/weights/best.pt").val(data=DATA_FALL, split="val", device=0)
    return {"map50": float(m.box.map50), "map5095": float(m.box.map)}

results_A, agg_A = run_over_seeds(stage_a_pipeline, out_dir="runs/seeds_stageA_fall")
print("\nPAPER (frame-level, fall):",
      f"mAP50 = {agg_A['map50']['mean']:.3f} +/- {agg_A['map50']['std']:.3f}",
      f"CI95 {[round(x,3) for x in agg_A['map50']['ci95']]}")

BEST_A = "runs/stageA_fall_s0/weights/best.pt"   # seed 0 feeds Stage B end-to-end
assert os.path.exists(BEST_A)

## [4] Assemble SENTRY and measure the parameter overhead

The TFM is materialized lazily on the first forward pass (it infers the pyramid
widths). **Reported: +21.1% parameters (5.47M).**

In [ ]:
# %% [4] SENTRY assembly + overhead
sentry_m = SentryYOLO(YOLO(BEST_A), hidden_ch=CFG["hidden_ch"]).cuda()
sentry_m(torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).cuda())   # materialize TFM

n_tfm = sum(p.numel() for p in sentry_m.tfm.parameters())
n_base = sum(p.numel() for p in sentry_m.base.parameters())
print(f"TFM: {n_tfm/1e6:.2f}M params (+{100*n_tfm/n_base:.1f}%)")
print("evidence terms (current frame):", sentry_m.last_evidence)

## [5] Stage B — supervised temporal fine-tuning of the TFM

**Design note (important).** Training the TFM with the temporal-consistency term
*alone* is ill-posed: its trivial optimum is "change nothing", which pins the
network at initialization (L_tc ≡ 0, zero gradient). Stage B is therefore driven
by the detector's supervised loss (`v8DetectionLoss`: box + cls + dfl), with L_tc
as an auxiliary regularizer — Eq. (3) of the paper.

Two integration details that the Ultralytics API requires: the loss needs the
`DetectionModel` itself (not its inner `Sequential`), and it reads `hyp.box` /
`hyp.cls` / `hyp.dfl` by **attribute** access, so `model.args` must be a
namespace, not a dict.

Windows are filtered to those containing ≥1 labeled frame (`event_only=True`),
which densifies supervision. Set `CFG["epochs_B"]` to 5, 10 or 30 to reproduce
the epoch ablation of Table 2.

In [ ]:
# %% [5] Stage B - supervised temporal fine-tuning, batched across clips
# Uses the tested implementation in sentry/train.py instead of an inline loop:
#   - build_criterion() delegates the loss CLASS choice to Ultralytics itself
#     (model.init_criterion()): v8DetectionLoss for YOLOv8/YOLO11,
#     E2ELoss for YOLO26 (dual-head, DFL-free, reg_max=1). No change needed
#     here if BEST_A comes from a different YOLO generation - verified live
#     for all three (see tests/test_train.py::test_stage_b_trains_across_yolo_generations).
#   - run_stage_b() batches the mini-batch's clips across the batch dimension
#     at each timestep: T model() calls per mini-batch instead of N*T. Measured
#     ~2x fewer calls / ~4x wall-clock on CPU with a tiny model (ARCHITECTURE.md
#     section 7.1); expect a comparable or larger reduction on the T4, plus the
#     memory benefit of automatic mixed precision (enabled below via use_amp).
#   - Raises RuntimeError instead of silently completing if an epoch has no
#     labeled frames (the earlier silent-failure mode this replaces).
SEED_B = 0
set_all_seeds(SEED_B)

sentry_m = SentryYOLO(YOLO(BEST_A), hidden_ch=CFG["hidden_ch"]).cuda()
sentry_m(torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).cuda())   # materialize TFM
sentry_m.train()            # training-format head output, required by the loss
sentry_m.freeze_base()      # backbone + neck frozen; only the TFM learns
for p in sentry_m.tfm.parameters():
    p.requires_grad_(True)

criterion = build_criterion(sentry_m.base, epochs=CFG["epochs_B"])
print("criterion:", type(criterion).__name__, "| nc =", sentry_m.base.model[-1].nc)

ids = json.load(open("/kaggle/working/le2i/splits.json"))["splits"]["train"]
g, winit = dataloader_seeding(SEED_B)
ds = VideoClipDataset("/kaggle/working/le2i/clips", ids, window=CFG["window"],
                      stride=CFG["window"], event_only=True)
dl = DataLoader(ds, batch_size=CFG["stageB_batch"], shuffle=True,
                collate_fn=collate_batched, generator=g, worker_init_fn=winit,
                num_workers=2, drop_last=True)
print(f"event windows: {len(ds)} (from {len(ids)} clips) | "
      f"batches/epoch: {len(dl)} (batch_size={CFG['stageB_batch']})")

history = run_stage_b(sentry_m, criterion, dl, epochs=CFG["epochs_B"], lr=1e-3,
                      lambda_tc=CFG["lambda_tc"], device=torch.device("cuda"),
                      use_amp=True)

os.makedirs("runs/sentry_stageB", exist_ok=True)
TFM_CKPT = f"runs/sentry_stageB/tfm_s0_e{CFG['epochs_B']}.pt"
torch.save(sentry_m.tfm.state_dict(), TFM_CKPT)
json.dump(history, open(f"runs/sentry_stageB/history_e{CFG['epochs_B']}.json", "w"), indent=1)
print("saved", TFM_CKPT)
# Expected pattern (Le2i, fall, yolov8m, epochs in {5,10,30}): `det` falls
# monotonically while TEST event F1 degrades - the overfitting evidence of
# Table 2. Re-confirm this holds under the batched loop: same loss, same data,
# same model, only the call structure changed, so the curve should match.

## [6] Level S evaluation — SENTRY vs frame-level (event F1)

Inference runs **once per clip** and the raw detections are cached; τ and n_c are
then swept over the cache in memory (they are post-hoc thresholds, so re-running
inference per combination would waste ~20× the GPU time).

τ and n_c are selected on **validation only**, frozen, then the test split
(Coffee_room_02, unseen camera) is scored once — the Phase-5 rule.

Set `LOAD_TFM=False` to evaluate the untrained TFM (the W=1-equivalent ablation,
which confirms the additive-skip property of Eq. 2).

**Reported (Table 2):** frame-level 0.727 · TFM untrained 0.621 · Stage B 5ep
0.056 · 10ep 0.132 · 30ep 0.047.

In [ ]:
# %% [6] Level S — cached inference + tau/n_c sweep + single test evaluation
import cv2
from itertools import product
from ultralytics.utils import ops
from ultralytics import YOLO as _YOLO

LOAD_TFM = True                     # False -> untrained-TFM ablation
CLIPS = "/kaggle/working/le2i/clips"
sp = json.load(open("/kaggle/working/le2i/splits.json"))["splits"]

def gt_events(split_ids):
    """One event per fall clip, spanning its labeled frames (index space of the
    sampled frame list, matching the inference loop)."""
    out = {}
    for cid in split_ids:
        frames = sorted(glob.glob(f"{CLIPS}/{cid}/frames/*.jpg"))
        fmap = {int(os.path.splitext(os.path.basename(f))[0]): i
                for i, f in enumerate(frames)}
        pos = [int(os.path.splitext(os.path.basename(l))[0])
               for l in sorted(glob.glob(f"{CLIPS}/{cid}/labels/*.txt"))
               if os.path.getsize(l) > 0]
        out[cid] = ([{"class_id": FALL, "t_start": fmap[min(pos)],
                      "t_end": fmap[max(pos)]}] if pos else [])
    return out
GT = gt_events(sp["val"] + sp["test"])

def infer_sentry(model, cids):
    cache = {}
    for cid in cids:
        model.eval(); model.reset_stream(); per_frame = []
        for fp in sorted(glob.glob(f"{CLIPS}/{cid}/frames/*.jpg")):
            im = cv2.resize(cv2.imread(fp), (CFG["imgsz"], CFG["imgsz"]))
            x = torch.from_numpy(im[:, :, ::-1].copy()).permute(2,0,1)[None].float().cuda()/255
            with torch.no_grad():
                raw = model(x)
                y = raw[0] if isinstance(raw, (list, tuple)) else raw
                if y.dim() == 3 and y.shape[1] >= 6:
                    d = ops.non_max_suppression(y, 0.03, 0.7, max_det=100)[0].cpu().numpy()
                    per_frame.append([{"bbox": r[:4].tolist(), "score": float(r[4]),
                                       "class_id": int(r[5])} for r in d])
                else:
                    per_frame.append([])
        cache[cid] = per_frame
        print(f"    infer {cid}: {len(per_frame)} frames", flush=True)
    return cache

def infer_baseline(yolo, cids):
    cache = {}
    for cid in cids:
        per_frame = []
        for fp in sorted(glob.glob(f"{CLIPS}/{cid}/frames/*.jpg")):
            r = yolo.predict(fp, conf=0.03, imgsz=CFG["imgsz"], verbose=False)[0]
            per_frame.append([{"bbox": [float(v) for v in b.xyxy[0]],
                               "score": float(b.conf), "class_id": int(b.cls)}
                              for b in r.boxes])
        cache[cid] = per_frame
        print(f"    infer {cid}: {len(per_frame)} frames", flush=True)
    return cache

def score(cache, cids, tau, n_c):
    allp, allg = [], []
    for cid in cids:
        lk = TubeLinker(iou_thr=0.5, max_gap=5)
        for t, dets in enumerate(cache[cid]):
            lk.update(t, [d for d in dets if d["score"] >= tau])
        allp += confirm_events(lk.finalize(), {FALL: n_c}); allg += GT[cid]
    return event_prf(allp, allg, tiou_thr=0.3), event_auc(allp, allg, tiou_thr=0.3)

def sweep(cache, cids):
    best = (-1, 0.3, 3)
    for tau, n_c in product([0.1, 0.2, 0.3, 0.4, 0.5], [2, 3, 5, 8]):
        prf, _ = score(cache, cids, tau, n_c)
        if prf[FALL]["F1"] > best[0]:
            best = (prf[FALL]["F1"], tau, n_c)
    return best

baseline = _YOLO(BEST_A)
sentry_e = SentryYOLO(YOLO(BEST_A), hidden_ch=CFG["hidden_ch"]).cuda()
sentry_e(torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).cuda())
if LOAD_TFM:
    sentry_e.tfm.load_state_dict(torch.load(TFM_CKPT))
    print("loaded", TFM_CKPT)
else:
    print("untrained-TFM ablation")

print("\ninferring on VAL..."); t0 = time.time()
cS_val, cB_val = infer_sentry(sentry_e, sp["val"]), infer_baseline(baseline, sp["val"])
sF1, sTau, sNc = sweep(cS_val, sp["val"]); print(f"SENTRY   val F1={sF1:.3f} @tau={sTau} n_c={sNc}")
bF1, bTau, bNc = sweep(cB_val, sp["val"]); print(f"baseline val F1={bF1:.3f} @tau={bTau} n_c={bNc}")

print("\ninferring on TEST (Coffee_room_02, unseen camera)...")
cS_te, cB_te = infer_sentry(sentry_e, sp["test"]), infer_baseline(baseline, sp["test"])
sP, sA = score(cS_te, sp["test"], sTau, sNc)
bP, bA = score(cB_te, sp["test"], bTau, bNc)
print("\n=== TEST ===")
print(f"SENTRY   : F1={sP[FALL]['F1']:.3f} P={sP[FALL]['P']:.3f} R={sP[FALL]['R']:.3f} "
      f"(tp={sP[FALL]['tp']} fp={sP[FALL]['fp']} fn={sP[FALL]['fn']})")
print(f"frame-lvl: F1={bP[FALL]['F1']:.3f} P={bP[FALL]['P']:.3f} R={bP[FALL]['R']:.3f} "
      f"(tp={bP[FALL]['tp']} fp={bP[FALL]['fp']} fn={bP[FALL]['fn']})")
print(f"\n>>> TEMPORAL DELTA (fall F1): {sP[FALL]['F1']-bP[FALL]['F1']:+.3f} <<<  "
      f"({time.time()-t0:.0f}s)")

tag = f"e{CFG['epochs_B']}" if LOAD_TFM else "untrained"
json.dump({"config": tag, "sentry": sP[FALL], "baseline": bP[FALL],
           "tau_nc_sentry": [sTau, sNc], "tau_nc_baseline": [bTau, bNc]},
          open(f"/kaggle/working/tierS_fall_{tag}.json", "w"), indent=1)
print(f"saved tierS_fall_{tag}.json")

## [7] Level E — UCF-Crime, zero-shot open-vocabulary, 3 prompt conditions

Reads the `.mp4` test videos directly (a recursive glob absorbs the mirror's
double nesting, `Anomaly-Videos-Part-2/Anomaly-Videos-Part-2/...`); no frame
extraction to disk. `Normal` test videos enter with an **empty** event list so
that false alarms on normal footage are penalized — omitting them inflates AUC.
Unmappable UCF classes (Arrest, Explosion) are excluded and declared.

Detections are cached once per condition (the vocabulary changes what the model
fires on, so each condition needs its own inference pass); τ and n_c are then
swept on a stratified **validation** half and reported on the held-out half.

**Reported (Table 3, 110 videos):** class-name AUC 0.000 / macro-F1 0.029 ·
compositional AUC 0.243 / macro-F1 0.215 · UCF-adapted AUC 0.128 / macro-F1 0.190.
Budget ≈ 35 min per condition.

In [ ]:
# %% [7] Level E — three prompt conditions
import cv2
from collections import defaultdict
from itertools import product
from ultralytics import YOLOWorld

MAP    = json.load(open(f"{REPO}/psid8/ucfcrime_mapping.json"))
P_ORIG = json.load(open(f"{REPO}/psid8/openvocab_prompts.json"))      # from the schema
P_UCF  = json.load(open(f"{REPO}/psid8/openvocab_prompts_ucf.json"))  # UCF-adapted

VIDX = {os.path.basename(fp): fp
        for fp in glob.glob(f"{CFG['ucf_root']}/**/*.mp4", recursive=True)}
print("videos indexed:", len(VIDX))          # reported: 1950

def parse_temporal(path):
    gts = {}
    for line in open(path):
        p = line.split()
        if len(p) < 6: continue
        vid, cls = p[0], p[1]
        if cls == "Normal" or vid.startswith("Normal"):
            gts[vid] = []                      # normals: FPs must be penalized
            continue
        tax = MAP.get(cls)
        if tax is None: continue               # Arrest, Explosion -> excluded
        gts[vid] = [{"class_id": ORDER.index(tax), "t_start": int(p[i]), "t_end": int(p[i+1])}
                    for i in (2, 4) if int(p[i]) >= 0]
    return gts

gts_all = parse_temporal(CFG["ucf_temporal"])
by_cls = defaultdict(list)
for vid, ev in gts_all.items():
    by_cls[ev[0]["class_id"] if ev else "normal"].append(vid)
keep = {v for vids in by_cls.values() for v in sorted(vids)[:CFG["ucf_per_class"]]}
GTE = {v: gts_all[v] for v in keep if VIDX.get(v)}
print(f"videos: {len(GTE)} ({CFG['ucf_per_class']}/class)")     # reported: 110

random.seed(0)
val_ids, test_ids = [], []
byc = defaultdict(list)
for v in GTE: byc[GTE[v][0]["class_id"] if GTE[v] else "normal"].append(v)
for c, vids in byc.items():
    random.shuffle(vids); h = len(vids) // 2
    val_ids += vids[:h]; test_ids += vids[h:]
print(f"val {len(val_ids)} | test {len(test_ids)}")             # reported: 53 | 57
MAPPED = sorted({GTE[v][0]["class_id"] for v in GTE if GTE[v]})

def build_vocab(pset, mode):
    vocab, owner = [], []
    for ci, name in enumerate(ORDER):
        ps = [pset[name]["class_prompt"]] if mode == "class" else pset[name]["compositional_prompts"]
        for p in ps:
            vocab.append(p); owner.append(ci)
    return vocab, owner

def cache_condition(vocab, owner):
    model = YOLOWorld("yolov8l-worldv2.pt"); model.set_classes(vocab)
    cache = {}
    for i, vid in enumerate(GTE):
        cap = cv2.VideoCapture(VIDX[vid]); pf = []; f = 0
        while True:
            if not cap.grab(): break
            f += 1
            if (f - 1) % CFG["fps_stride"]: continue    # cheap frame skip
            ok, frame = cap.retrieve()
            if not ok: break
            r = model.predict(frame, conf=0.02, verbose=False)[0]   # low floor; swept later
            pf.append([{"bbox": [float(v) for v in b.xyxy[0]], "score": float(b.conf),
                        "class_id": owner[int(b.cls)]} for b in r.boxes])
        cap.release(); cache[vid] = pf
        if (i + 1) % 15 == 0: print(f"    cached {i+1}/{len(GTE)}", flush=True)
    return cache

def score_E(cache, vids, tau, n_c):
    allp, allg = [], []
    for vid in vids:
        lk = TubeLinker(iou_thr=0.3, max_gap=3)
        for t, dets in enumerate(cache[vid]):
            lk.update(t, [d for d in dets if d["score"] >= tau])
        al = confirm_events(lk.finalize(), {c: n_c for c in range(8)})
        for a in al:                                    # tube time -> frame index
            a["t_start"] *= CFG["fps_stride"]; a["t_end"] *= CFG["fps_stride"]
        allp += al; allg += GTE[vid]
    return event_prf(allp, allg, 0.3), event_auc(allp, allg, 0.3)

def sweep_and_test(cache, tag):
    best = (-1, 0.1, 5)
    for tau, n_c in product([0.05, 0.1, 0.15, 0.2, 0.3], [3, 5, 10, 25, 50]):
        prf, _ = score_E(cache, val_ids, tau, n_c)
        mf1 = sum(prf[c]["F1"] for c in MAPPED) / len(MAPPED)
        if mf1 > best[0]: best = (mf1, tau, n_c)
    prf, auc = score_E(cache, test_ids, best[1], best[2])
    macro = sum(prf[c]["F1"] for c in MAPPED) / len(MAPPED)
    print(f"\n[{tag}] AUC={auc:.3f} macro-F1={macro:.3f} @tau={best[1]} n_c={best[2]}")
    for c in MAPPED:
        m = prf[c]
        print(f"  {ORDER[c]:12s} P={m['P']:.3f} R={m['R']:.3f} F1={m['F1']:.3f}")
    return {"auc": auc, "macro_f1": macro, "tau": best[1], "n_c": best[2],
            "prf": {ORDER[c]: prf[c] for c in MAPPED}}

resultsE = {}
for tag, pset, mode in [("class", P_ORIG, "class"),
                        ("comp_schema", P_ORIG, "comp"),
                        ("comp_ucf", P_UCF, "comp")]:
    print(f"\n=== caching [{tag}] ===", flush=True); t0 = time.time()
    v, o = build_vocab(pset, mode)
    resultsE[tag] = sweep_and_test(cache_condition(v, o), tag)
    print(f"  ({time.time()-t0:.0f}s)")
    json.dump(resultsE, open("/kaggle/working/tierE_expanded.json", "w"), indent=1)

print("\n=== FINAL COMPARISON (Table 3) ===")
for tag, r in resultsE.items():
    print(f"{tag:12s} AUC={r['auc']:.3f} macro-F1={r['macro_f1']:.3f}")

## [8] Latency — frame-level vs SENTRY (1×T4, batch=1, FP16)

50 warm-up iterations, then the mean over 500 forwards.
**Reported:** frame-level 12.47 ms (80.2 FPS) · SENTRY 15.46 ms (64.7 FPS) ·
TFM overhead +2.99 ms (+24.0%) · 2 concurrent 25-fps streams per GPU.

In [ ]:
# %% [8] Latency measurement
from ultralytics import YOLO as _YOLO
def measure(model, n_warm=50, n_iter=500):
    x = torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).half().cuda()
    for _ in range(n_warm):
        with torch.no_grad(): model(x)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(n_iter):
        with torch.no_grad(): model(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter * 1000

base_lat = _YOLO(BEST_A).model.eval().half().cuda()
ms_base = measure(base_lat)

sentry_l = SentryYOLO(YOLO(BEST_A), hidden_ch=CFG["hidden_ch"]).cuda()
sentry_l(torch.zeros(1, 3, CFG["imgsz"], CFG["imgsz"]).cuda())
sentry_l.eval().half(); sentry_l.reset_stream()
ms_sentry = measure(sentry_l)

print(f"frame-level: {ms_base:.2f} ms/frame | {1000/ms_base:.1f} FPS")
print(f"SENTRY     : {ms_sentry:.2f} ms/frame | {1000/ms_sentry:.1f} FPS | "
      f"25-fps streams/GPU: {streams_per_gpu(1000/ms_sentry)}")
print(f"TFM overhead: +{ms_sentry-ms_base:.2f} ms "
      f"({100*(ms_sentry-ms_base)/ms_base:.1f}%)")

json.dump({"frame_level_ms": ms_base, "sentry_ms": ms_sentry,
           "overhead_pct": 100*(ms_sentry-ms_base)/ms_base,
           "sentry_fps": 1000/ms_sentry,
           "streams_per_gpu": streams_per_gpu(1000/ms_sentry)},
          open("/kaggle/working/latency.json", "w"), indent=1)

## [8b] Figures — training curves, overfitting ablation, confusion matrix

Ultralytics already writes `results.png` per Stage-A run. These add the figures it
cannot produce: the Stage-B curves, the epoch-ablation plot that *is* the paper's
overfitting evidence, the event-level confusion matrix, and the Level-E prompt
comparison.

In [ ]:
# %% [8b] Figures
from sentry.plots import (plot_stageb_curves, plot_epoch_ablation, plot_confusion_matrix,
                          plot_prompt_comparison, plot_seed_variance, parse_stageb_log)
from sentry.metrics import event_confusion_matrix
from IPython.display import Image, display

FIG = "/kaggle/working/figures"; os.makedirs(FIG, exist_ok=True)

# --- Stage A: per-seed mAP with bootstrap CI ---
agg = json.load(open("runs/seeds_stageA_fall/aggregate.json"))
display(Image(plot_seed_variance(agg, "map50", out=f"{FIG}/stageA_seeds.png")))
# Ultralytics' own curves live in runs/stageA_fall_s0/results.png
if os.path.exists("runs/stageA_fall_s0/results.png"):
    display(Image("runs/stageA_fall_s0/results.png"))

# --- Stage B curves: paste the epoch lines printed by cell [5] ---
STAGEB_LOG = """
epoch 0: det=15.564 | tc=0.7603
epoch 1: det=15.024 | tc=0.5767
epoch 2: det=14.827 | tc=0.5315
epoch 3: det=14.660 | tc=0.5161
epoch 4: det=14.596 | tc=0.4949
epoch 5: det=14.455 | tc=0.5285
epoch 6: det=14.302 | tc=0.5609
epoch 7: det=14.321 | tc=0.5104
epoch 8: det=14.218 | tc=0.5056
epoch 9: det=14.103 | tc=0.4929
"""
hist = parse_stageb_log(STAGEB_LOG)
if hist:
    display(Image(plot_stageb_curves(hist, out=f"{FIG}/stageB_curves.png")))

# --- THE overfitting figure: fill one row per Stage-B run you evaluated ---
ablation = [
    {"epochs": 0,  "train_det": None, "test_f1": 0.621, "fp": 8},    # untrained TFM
    {"epochs": 5,  "train_det": 16.7, "test_f1": 0.056, "fp": 407},
    {"epochs": 10, "train_det": 14.1, "test_f1": 0.132, "fp": 158},
    {"epochs": 30, "train_det": 12.7, "test_f1": 0.047, "fp": 488},
]
BASELINE_F1 = 0.727
display(Image(plot_epoch_ablation(ablation, BASELINE_F1, out=f"{FIG}/epoch_ablation.png")))

# --- Event-level confusion matrix (Level S, test split) ---
# Rebuild alerts with the frozen tau/n_c, then compare against GT.
alerts_S, gts_S = [], []
for cid in sp["test"]:
    lk = TubeLinker(iou_thr=0.5, max_gap=5)
    for t, dets in enumerate(cS_te[cid]):
        lk.update(t, [d for d in dets if d["score"] >= sTau])
    alerts_S += confirm_events(lk.finalize(), {FALL: sNc}); gts_S += GT[cid]
cm_S = event_confusion_matrix(alerts_S, gts_S, n_classes=8, tiou_thr=0.3)
display(Image(plot_confusion_matrix(cm_S, out=f"{FIG}/confusion_levelS.png",
                                    title="Level S (fall), test split - SENTRY")))

# --- Level E: prompt-condition comparison + confusion matrix ---
if "resultsE" in dir():
    display(Image(plot_prompt_comparison(resultsE, out=f"{FIG}/prompt_comparison.png")))

print("figures written to", FIG)

## [9] Export artifacts

Every number in the paper must be traceable to one of these files. Download the
bundle and attach it to the Zenodo deposit alongside the code.

In [ ]:
# %% [9] Export
import shutil
OUT = "/kaggle/working/artifacts"
os.makedirs(OUT, exist_ok=True)
for pat in ["versions.json", "/kaggle/working/tierS_fall_*.json",
            "/kaggle/working/tierE_*.json", "/kaggle/working/latency.json",
            "/kaggle/working/le2i/manifest.json", "/kaggle/working/le2i/splits.json",
            "/kaggle/working/le2i/integrity_report.json",
            "runs/seeds_stageA_fall/*.json"]:
    for f in glob.glob(pat):
        shutil.copy(f, OUT)
for s in (0, 1, 2):
    w = f"runs/stageA_fall_s{s}/weights/best.pt"
    if os.path.exists(w): shutil.copy(w, f"{OUT}/best_stageA_s{s}.pt")
for w in glob.glob("runs/sentry_stageB/*.pt"):
    shutil.copy(w, OUT)

!cd /kaggle/working && zip -qr artifacts.zip artifacts && ls -lh artifacts.zip
print("\nContents:"); print("\n".join(sorted(os.listdir(OUT))))
print("\nDownload artifacts.zip; it is the evidence trail for every reported number.")